In [1]:
import sys, os
sys.path.append(os.getcwd() + '/../')

Setup & Dependencies

First, we ensure the necessary libraries are installed and we create the "easy job" target file (my_math.py) directly from the notebook.

In [ ]:
# Create the target file we want to test
# usage of %%writefile saves this cell's content to a file named 'my_math.py'
import os

code_content = """
def add(a, b):
    return a + b

def is_prime(n):
    if n <= 1: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
"""

with open("my_math.py", "w") as f:
    f.write(code_content)

print("Created 'my_math.py' successfully.")

Created 'my_math.py' successfully.


Cell 2: Step 1 - Generate the Tests (The Generator)

This cell loads the model and uses Syncode to write the test file.

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from syncode import Syncode

# 1. Configuration
model_name = "microsoft/phi-2" # Or your preferred model
output_filename = "test_generated.py"

# 2. Load Model & Syncode
print("Loading model... (this may take a minute)")
syn_llm = Syncode(model = model_name, mode="grammar_strict", grammar="python", max_new_tokens=200)


Loading model... (this may take a minute)
[2026-01-22 16:07:14,687-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.69s/it]


[2026-01-22 16:07:32,420-accelerate.big_modeling] - Some parameters are on the meta device because they were offloaded to the disk.


In [ ]:
# 3. Generate
print("Generating tests...")
prompt = "import unittest\nfrom my_math import add, is_prime\n\nclass TestMath(unittest.TestCase):\n"
custom_stop_words = ["Exercise:", "\n\n\n", "unittest.main()"]
generated_output = syn_llm.infer(prompt, stop_words=custom_stop_words)[0]

# 4. Save the Result
full_code = prompt + generated_output
with open(output_filename, "w") as f:
    f.write(full_code)

print(f"Step 1 Complete: generated code saved to '{output_filename}'")
print("-" * 20)
print(full_code)

Generating tests...
Step 1 Complete: generated code saved to 'test_generated.py'
--------------------
import unittest
from my_math import add, is_prime

class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(2, 3), 5)
        self.assertEqual(add(-2, 3), 1)
        self.assertEqual(add(0, 0), 0)
        
    def test_is_prime(self):
        self.assertTrue(is_prime(2))
        self.assertTrue(is_prime(3))
        self.assertTrue(is_prime(5))
        self.assertFalse(is_prime(4))
        self.assertFalse(is_prime(6))

if __name__ == '__main__':
    unittest.main()


Cell 3: Step 2 - The Meta-Test (The Verification)

This cell acts as the "Inspector." It programmatically loads the file generated in Step 1 and runs it.

Note for Jupyter: We use TextTestRunner explicitly instead of unittest.main() because unittest.main() will try to exit the kernel when it finishes, which we want to avoid.

In [9]:
import unittest
import sys

class TestTheGeneratedTests(unittest.TestCase):
    """
    This is the META-TEST suite.
    It does not test math; it tests if the GENERATED file works.
    """
    
    def test_generated_file_validity(self):
        """Phase 1: syntax check"""
        try:
            with open("test_generated.py", "r") as f:
                compile(f.read(), "test_generated.py", "exec")
        except IOError:
            self.fail("The generated file does not exist!")
        except SyntaxError:
            self.fail("The generated code has syntax errors!")

    def test_generated_tests_pass(self):
        """Phase 2: Execution check"""
        # Dynamically import the generated file
        if "test_generated" in sys.modules:
            del sys.modules["test_generated"] # Reload if it already exists
        import test_generated

        # Load tests from that module
        loader = unittest.TestLoader()
        suite = loader.loadTestsFromModule(test_generated)
        
        # Run them silently (verbosity=1 to see dots, 0 for silence)
        runner = unittest.TextTestRunner(stream=sys.stdout, verbosity=1)
        result = runner.run(suite)
        
        # The crucial check: Did the generated tests pass?
        if not result.wasSuccessful():
            self.fail(f"Generated tests failed! Errors: {len(result.errors)}, Failures: {len(result.failures)}")

# Run the Meta-Test
# argv=['first-arg-is-ignored'] is required because Jupyter passes kernel args that unittest doesn't understand
# exit=False prevents the notebook from shutting down after the test
unittest.main(argv=['first-arg-is-ignored'], exit=False)

.

..
----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


.
----------------------------------------------------------------------
Ran 2 tests in 0.019s

OK
